# Phase-2 Training Pipeline — b23cm1036
**Binary Age Classification: ResNet-18 + CLIP Feature Distillation**

Strategy:
- Stage 1: CLIP ViT-L/14 distillation on ALL data (backbone learning)
- Stage 2: Classification on TRAIN only (with validation, save best EMA)
- Stage 3: Final training on ALL data (from best checkpoint)

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install CLIP
!pip install git+https://github.com/openai/CLIP.git -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


In [3]:
# Imports
import os, sys, csv, copy, math, random, time
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm

In [4]:
# CONFIGURATION
ROLL_NO = "b23cm1036"

# Drive paths
project_path = "/content/drive/MyDrive/DL Assignment"
checkpoint_dir = os.path.join(project_path, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)

DATA_ROOT = os.path.join(project_path, "dataset")
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "valid")
VAL_CSV   = os.path.join(DATA_ROOT, "valid_labels.csv")

sys.path.insert(0, project_path)

# CLIP teacher
CLIP_MODEL_NAME = "ViT-L/14"
CLIP_FEAT_DIM = 768
CLIP_CACHE_FILE = os.path.join(checkpoint_dir, "clip_embeddings_vitl14.pt")

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 42

# Stage 1: CLIP Distillation
DISTILL_EPOCHS = 50
DISTILL_LR = 1e-3
DISTILL_WARMUP = 5

# Stage 2: Classification Phase 1 (train only, with validation)
CLASS_EPOCHS_P1 = 15
CLASS_LR_BACKBONE = 1e-4
CLASS_LR_HEAD = 1e-3
CLASS_WARMUP = 3

# Stage 3: Final training (all data)
CLASS_EPOCHS_P2 = 10
CLASS_LR_BACKBONE_P2 = 5e-5
CLASS_LR_HEAD_P2 = 5e-4

# EMA / Mixup / Regularization
EMA_DECAY = 0.999
MIXUP_ALPHA = 0.2
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

# Checkpoint paths
BACKBONE_BEST_PT   = os.path.join(checkpoint_dir, "backbone_best.pt")
BACKBONE_DISTILLED  = os.path.join(checkpoint_dir, "backbone_distilled.pth")
BEST_PHASE1_PT     = os.path.join(checkpoint_dir, "best_phase1.pt")
FINAL_OUT          = os.path.join(project_path, f"{ROLL_NO}.pth")

print(f"Project path: {project_path}")
print(f"Dataset: {DATA_ROOT}")
print(f"Checkpoints: {checkpoint_dir}")

Project path: /content/drive/MyDrive/DL Assignment
Dataset: /content/drive/MyDrive/DL Assignment/dataset
Checkpoints: /content/drive/MyDrive/DL Assignment/checkpoints


In [5]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
else:
    print("WARNING: No GPU — enable GPU in Runtime > Change runtime type")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# IMPORT MODEL CLASS
from b23cm1036 import build_model, MyAgeClassifier

model = build_model(num_classes=2).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

Model parameters: 11,309,890


In [7]:
# DATASET & DATA LOADING
class FaceDataset(Dataset):
    """Returns (image, label, global_index) for CLIP embedding lookup."""
    def __init__(self, paths_labels, transform=None):
        self.data = paths_labels
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, idx


def load_train_data():
    samples = []
    for label in [0, 1]:
        cls_dir = os.path.join(TRAIN_DIR, str(label))
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith(".png"):
                samples.append((os.path.join(cls_dir, fname), label))
    return samples


def load_val_data():
    samples = []
    with open(VAL_CSV) as f:
        for row in csv.DictReader(f):
            path = os.path.join(VAL_DIR, row["image"])
            samples.append((path, int(row["label"])))
    return samples


train_data = load_train_data()
val_data = load_val_data()
all_data = train_data + val_data

print(f"Train: {len(train_data)} images")
print(f"Val:   {len(val_data)} images")
print(f"Total: {len(all_data)} images")

Train: 18332 images
Val:   134 images
Total: 18466 images


In [8]:
# TRANSFORMS
# Stage 1: Mild augmentation for distillation
distill_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Stage 2/3: Moderate augmentation for classification
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [9]:
# CLIP EMBEDDING PRECOMPUTATION
def precompute_clip_embeddings(all_paths):
    if os.path.exists(CLIP_CACHE_FILE):
        print(f"Loading cached CLIP embeddings from {CLIP_CACHE_FILE}")
        data = torch.load(CLIP_CACHE_FILE, map_location="cpu", weights_only=True)
        if data.shape[0] == len(all_paths):
            print(f"  Shape: {data.shape}")
            return data
        print(f"  Cache mismatch ({data.shape[0]} vs {len(all_paths)}), recomputing...")

    import clip
    print(f"Loading CLIP {CLIP_MODEL_NAME}...")
    clip_model, preprocess = clip.load(CLIP_MODEL_NAME, device=device)
    clip_model.eval()

    all_embeddings = []
    extract_bs = 32

    for i in tqdm(range(0, len(all_paths), extract_bs), desc="CLIP embeddings"):
        batch_paths = all_paths[i : i + extract_bs]
        images = [preprocess(Image.open(p).convert("RGB")) for p in batch_paths]
        batch = torch.stack(images).to(device)

        with torch.no_grad():
            features = clip_model.encode_image(batch).float()
            features = F.normalize(features, dim=-1)

        all_embeddings.append(features.cpu())

    embeddings = torch.cat(all_embeddings, dim=0)
    torch.save(embeddings, CLIP_CACHE_FILE)
    print(f"Saved: {embeddings.shape} -> {CLIP_CACHE_FILE}")

    del clip_model
    torch.cuda.empty_cache()
    return embeddings


all_paths = [p for p, _ in all_data]
clip_embeddings = precompute_clip_embeddings(all_paths)

Loading cached CLIP embeddings from /content/drive/MyDrive/DL Assignment/checkpoints/clip_embeddings_vitl14.pt
  Shape: torch.Size([18466, 768])


In [10]:
# HELPER CLASSES & FUNCTIONS
# --- EMA ---
class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()

    def update(self, model):
        with torch.no_grad():
            for ema_p, model_p in zip(self.shadow.parameters(), model.parameters()):
                ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)
            for ema_b, model_b in zip(self.shadow.buffers(), model.buffers()):
                ema_b.data.copy_(model_b.data)

    def apply_to(self, model):
        for p, ema_p in zip(model.parameters(), self.shadow.parameters()):
            p.data.copy_(ema_p.data)
        for b, ema_b in zip(model.buffers(), self.shadow.buffers()):
            b.data.copy_(ema_b.data)


# --- Mixup ---
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# --- Cosine LR with warmup ---
def cosine_lr(epoch, warmup, peak_lr, total_epochs):
    if epoch < warmup:
        return peak_lr * (epoch + 1) / warmup
    t = (epoch - warmup) / max(total_epochs - warmup, 1)
    return 1e-6 + 0.5 * (peak_lr - 1e-6) * (1 + math.cos(math.pi * t))


# --- Evaluation ---
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return 100.0 * correct / total if total > 0 else 0.0

In [11]:
# STAGE 1: CLIP FEATURE DISTILLATION (50 epochs here, 150 in the original script)

# Check if we already have a distilled backbone saved
if os.path.exists(BACKBONE_DISTILLED):
    print(f"Found distilled backbone at {BACKBONE_DISTILLED}, loading...")
    model = torch.load(BACKBONE_DISTILLED, map_location=device, weights_only=False)
    model.to(device)
    print("Backbone loaded — skip to Stage 2!")
else:
    print(f"No cached backbone found, running Stage 1 distillation...")

    epochs = DISTILL_EPOCHS
    lr = DISTILL_LR
    warmup = DISTILL_WARMUP

    print(f"\n{'=' * 60}")
    print(f"  STAGE 1: CLIP Feature Distillation ({epochs} epochs)")
    print(f"{'=' * 60}")

    # Projection head: 512 -> CLIP dim
    proj = nn.Sequential(
        nn.Linear(512, 768),
        nn.GELU(),
        nn.Linear(768, CLIP_FEAT_DIM),
    ).to(device)

    dataset_s1 = FaceDataset(all_data, transform=distill_transform)
    loader_s1 = DataLoader(
        dataset_s1, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    )

    optimizer_s1 = optim.AdamW(
        list(model.parameters()) + list(proj.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY,
    )

    best_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        proj.train()

        cur_lr = cosine_lr(epoch, warmup, lr, epochs)
        for g in optimizer_s1.param_groups:
            g["lr"] = cur_lr

        running_loss, n_samples = 0.0, 0

        pbar = tqdm(loader_s1, desc=f"Distill {epoch+1:03d}/{epochs}", leave=False)
        for imgs, _, idxs in pbar:
            imgs = imgs.to(device)
            targets = clip_embeddings[idxs].to(device)

            optimizer_s1.zero_grad()
            features = model.extract_features(imgs)
            projected = F.normalize(proj(features), dim=-1)

            loss = (1 - F.cosine_similarity(projected, targets, dim=-1)).mean()
            loss.backward()

            nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(proj.parameters()), 1.0
            )
            optimizer_s1.step()

            running_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)
            pbar.set_postfix(loss=f"{running_loss / n_samples:.4f}")

        avg_loss = running_loss / n_samples
        tag = ""
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), BACKBONE_BEST_PT)
            tag = " *"

        if (epoch + 1) % 5 == 0 or epoch == 0 or tag:
            print(
                f"  Ep {epoch+1:03d}/{epochs} | "
                f"Loss: {avg_loss:.4f} | Best: {best_loss:.4f} | "
                f"LR: {cur_lr:.6f}{tag}"
            )

    # Restore best backbone
    model.load_state_dict(
        torch.load(BACKBONE_BEST_PT, map_location=device, weights_only=True)
    )

    # Save distilled backbone to Drive
    torch.save(model, BACKBONE_DISTILLED)
    print(f"\nDistillation done! Best loss: {best_loss:.4f}")
    print(f"Saved backbone -> {BACKBONE_DISTILLED}")

    del proj, loader_s1, dataset_s1
    torch.cuda.empty_cache()

No cached backbone found, running Stage 1 distillation...

  STAGE 1: CLIP Feature Distillation (50 epochs)


Distill 001/50:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 001/50 | Loss: 0.2148 | Best: 0.2148 | LR: 0.000200 *


Distill 002/50:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 002/50 | Loss: 0.1803 | Best: 0.1803 | LR: 0.000400 *


...

  Ep 048/50 | Loss: 0.0673 | Best: 0.0673 | LR: 0.000012 *


Distill 049/50:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 049/50 | Loss: 0.0671 | Best: 0.0671 | LR: 0.000006 *


Distill 050/50:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 050/50 | Loss: 0.0671 | Best: 0.0671 | LR: 0.000002 *

Distillation done! Best loss: 0.0671
Saved backbone -> /content/drive/MyDrive/DL Assignment/checkpoints/backbone_distilled.pth


In [12]:
# Free CLIP embeddings (no longer needed)
del clip_embeddings1
torch.cuda.empty_cache()
print("CLIP embeddings freed.")

CLIP embeddings freed.


In [13]:
# STAGE 2: CLASSIFICATION PHASE 1 (train only, validate)
epochs_p1 = CLASS_EPOCHS_P1
lr_bb = CLASS_LR_BACKBONE
lr_hd = CLASS_LR_HEAD
warmup = CLASS_WARMUP

print(f"\n{'=' * 60}")
print(f"  STAGE 2: Classification Phase 1 ({epochs_p1} epochs)")
print(f"  Train: {len(train_data)} | Val: {len(val_data)}")
print(f"{'=' * 60}")

train_ds = FaceDataset(train_data, transform=train_transform)
val_ds = FaceDataset(val_data, transform=val_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Differential learning rates
head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]

optimizer_p1 = optim.AdamW([
    {"params": backbone_params, "lr": lr_bb},
    {"params": head_params, "lr": lr_hd},
], weight_decay=WEIGHT_DECAY)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

ema = EMAModel(model, decay=EMA_DECAY)
best_val_acc = 0.0

for epoch in range(epochs_p1):
    model.train()

    cur_lr_bb = cosine_lr(epoch, warmup, lr_bb, epochs_p1)
    cur_lr_hd = cosine_lr(epoch, warmup, lr_hd, epochs_p1)
    optimizer_p1.param_groups[0]["lr"] = cur_lr_bb
    optimizer_p1.param_groups[1]["lr"] = cur_lr_hd

    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"P1 Ep {epoch+1:02d}/{epochs_p1}", leave=False)
    for imgs, labels, _ in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)

        optimizer_p1.zero_grad()
        logits = model(mixed_imgs)
        loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_p1.step()
        ema.update(model)

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (
            lam * (preds == y_a).float().sum()
            + (1 - lam) * (preds == y_b).float().sum()
        ).item()
        total += imgs.size(0)

        pbar.set_postfix(
            loss=f"{running_loss / total:.4f}",
            acc=f"{100 * correct / total:.1f}%",
        )

    train_acc = 100.0 * correct / total
    train_loss = running_loss / total

    # Validate using EMA model
    val_acc = evaluate(ema.shadow, val_loader)

    tag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(ema.shadow.state_dict(), BEST_PHASE1_PT)
        tag = " *"

    print(
        f"  Ep {epoch+1:02d}/{epochs_p1} | "
        f"Loss: {train_loss:.4f} | Train: {train_acc:.1f}% | "
        f"Val(EMA): {val_acc:.1f}% | Best: {best_val_acc:.1f}%{tag}"
    )

# Restore best EMA checkpoint
model.load_state_dict(
    torch.load(BEST_PHASE1_PT, map_location=device, weights_only=True)
)

del train_loader, val_loader, train_ds, val_ds
torch.cuda.empty_cache()

print(f"\nPhase 1 done! Best Val Acc: {best_val_acc:.1f}%")


  STAGE 2: Classification Phase 1 (15 epochs)
  Train: 18332 | Val: 134


P1 Ep 01/15:   0%|          | 0/286 [00:00<?, ?it/s]

  Ep 01/15 | Loss: 0.3545 | Train: 93.1% | Val(EMA): 75.4% | Best: 75.4% *


P1 Ep 02/15:   0%|          | 0/286 [00:00<?, ?it/s]

  Ep 02/15 | Loss: 0.3323 | Train: 94.2% | Val(EMA): 71.6% | Best: 75.4%


...

  Ep 13/15 | Loss: 0.3044 | Train: 94.8% | Val(EMA): 71.6% | Best: 75.4%


P1 Ep 14/15:   0%|          | 0/286 [00:00<?, ?it/s]

  Ep 14/15 | Loss: 0.2965 | Train: 95.0% | Val(EMA): 72.4% | Best: 75.4%


P1 Ep 15/15:   0%|          | 0/286 [00:00<?, ?it/s]

  Ep 15/15 | Loss: 0.3106 | Train: 94.2% | Val(EMA): 73.9% | Best: 75.4%

Phase 1 done! Best Val Acc: 75.4%


In [14]:
# STAGE 3: FINAL TRAINING ON ALL DATA (10 epochs here, 15 in the original script)
epochs_p2 = CLASS_EPOCHS_P2
lr_bb_f = CLASS_LR_BACKBONE_P2
lr_hd_f = CLASS_LR_HEAD_P2

print(f"\n{'=' * 60}")
print(f"  STAGE 3: Final Training on All Data ({epochs_p2} epochs)")
print(f"  Images: {len(all_data)}")
print(f"{'=' * 60}")

final_ds = FaceDataset(all_data, transform=train_transform)
final_loader = DataLoader(
    final_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)

head_params = [p for n, p in model.named_parameters() if n.startswith("classifier")]
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]

optimizer_f = optim.AdamW([
    {"params": backbone_params, "lr": lr_bb_f},
    {"params": head_params, "lr": lr_hd_f},
], weight_decay=WEIGHT_DECAY)

criterion_f = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
ema_f = EMAModel(model, decay=EMA_DECAY)

for epoch in range(epochs_p2):
    model.train()

    cur_lr_bb = cosine_lr(epoch, 2, lr_bb_f, epochs_p2)
    cur_lr_hd = cosine_lr(epoch, 2, lr_hd_f, epochs_p2)
    optimizer_f.param_groups[0]["lr"] = cur_lr_bb
    optimizer_f.param_groups[1]["lr"] = cur_lr_hd

    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(final_loader, desc=f"Final {epoch+1:02d}/{epochs_p2}", leave=False)
    for imgs, labels, _ in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)

        optimizer_f.zero_grad()
        logits = model(mixed_imgs)
        loss = mixup_criterion(criterion_f, logits, y_a, y_b, lam)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_f.step()
        ema_f.update(model)

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (
            lam * (preds == y_a).float().sum()
            + (1 - lam) * (preds == y_b).float().sum()
        ).item()
        total += imgs.size(0)

        pbar.set_postfix(
            loss=f"{running_loss / total:.4f}",
            acc=f"{100 * correct / total:.1f}%",
        )

    train_acc = 100.0 * correct / total
    print(
        f"  Ep {epoch+1:02d}/{epochs_p2} | "
        f"Loss: {running_loss / total:.4f} | Acc: {train_acc:.1f}%"
    )

# Apply EMA weights
ema_f.apply_to(model)

del final_loader, final_ds
torch.cuda.empty_cache()

print("\nFinal training done!")


  STAGE 3: Final Training on All Data (10 epochs)
  Images: 18466


Final 01/10:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 01/10 | Loss: 0.3634 | Acc: 92.6%


Final 02/10:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 02/10 | Loss: 0.3487 | Acc: 93.1%


...

  Ep 08/10 | Loss: 0.3281 | Acc: 93.8%


Final 09/10:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 09/10 | Loss: 0.3231 | Acc: 94.1%


Final 10/10:   0%|          | 0/288 [00:00<?, ?it/s]

  Ep 10/10 | Loss: 0.3325 | Acc: 93.6%

Final training done!


In [15]:
# FINAL EVALUATION & SAVE
print(f"\n{'=' * 60}")
print(f"  FINAL EVALUATION")
print(f"{'=' * 60}")

val_ds_eval = FaceDataset(val_data, transform=val_transform)
val_loader_eval = DataLoader(
    val_ds_eval, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
final_val_acc = evaluate(model, val_loader_eval)

train_ds_eval = FaceDataset(train_data, transform=val_transform)
train_loader_eval = DataLoader(
    train_ds_eval, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
final_train_acc = evaluate(model, train_loader_eval)

print(f"  Train Accuracy:       {final_train_acc:.2f}%")
print(f"  Validation Accuracy:  {final_val_acc:.2f}%")
print(f"  Best Phase-1 Val Acc: {best_val_acc:.2f}%")

# Save final model
torch.save(model, FINAL_OUT)
size_mb = os.path.getsize(FINAL_OUT) / 1e6
print(f"\nSaved final model -> {FINAL_OUT} ({size_mb:.1f} MB)")


  FINAL EVALUATION
  Train Accuracy:       99.75%
  Validation Accuracy:  84.33%
  Best Phase-1 Val Acc: 75.37%

Saved final model -> /content/drive/MyDrive/DL Assignment/b23cm1036.pth (45.3 MB)
